<a href="https://colab.research.google.com/github/ayangelah/GCG-toy-experiment/blob/main/GCG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
model.eval()

prompt = "The capital of France is"
suffix = " blue green red"
target = " Paris"

prompt_ids = tokenizer(
    prompt,
    return_tensors="pt",
    add_special_tokens=False
).input_ids.to(device)

suffix_ids = tokenizer(
    suffix,
    return_tensors="pt",
    add_special_tokens=False
).input_ids.to(device)

target_ids = tokenizer(
    target,
    return_tensors="pt",
    add_special_tokens=False
).input_ids.to(device)

print("prompt:", tokenizer.convert_ids_to_tokens(prompt_ids[0]))
print("suffix:", tokenizer.convert_ids_to_tokens(suffix_ids[0]))
print("target:", tokenizer.convert_ids_to_tokens(target_ids[0]))

assert target_ids.shape[1] == 1

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

prompt: ['The', 'Ġcapital', 'Ġof', 'ĠFrance', 'Ġis']
suffix: ['Ġblue', 'Ġgreen', 'Ġred']
target: ['ĠParis']


In [2]:
full_ids = torch.cat([prompt_ids, suffix_ids], dim=1)

suffix_start = prompt_ids.shape[1]

print(tokenizer.convert_ids_to_tokens(full_ids[0]))
print("suffix begins at index:", suffix_start)

['The', 'Ġcapital', 'Ġof', 'ĠFrance', 'Ġis', 'Ġblue', 'Ġgreen', 'Ġred']
suffix begins at index: 5


In [3]:
edit_position = full_ids.shape[1] - 1

print(
    "editing:",
    tokenizer.decode([full_ids[0, edit_position].item()])
)

editing:  red


In [4]:
def get_target_loss(input_ids):
    with torch.no_grad():
        outputs = model(input_ids=input_ids)

        next_token_logits = outputs.logits[:, -1, :]

        loss = F.cross_entropy(
            next_token_logits,
            target_ids[:, 0]
        )

    return loss.item()

In [8]:
baseline_loss = get_target_loss(full_ids)

In [9]:
embedding_layer = model.get_input_embeddings()

inputs_embeds = embedding_layer(full_ids).detach()
inputs_embeds.requires_grad_(True)

outputs = model(inputs_embeds=inputs_embeds)

next_token_logits = outputs.logits[:, -1, :]

loss = F.cross_entropy(
    next_token_logits,
    target_ids[:, 0]
)

model.zero_grad()

loss.backward()

In [10]:
g = inputs_embeds.grad[0, edit_position]

print("inputs_embeds:", inputs_embeds.shape)
print("all input gradients:", inputs_embeds.grad.shape)
print("g:", g.shape)

inputs_embeds: torch.Size([1, 8, 768])
all input gradients: torch.Size([1, 8, 768])
g: torch.Size([768])


In [11]:
E = embedding_layer.weight.detach()

scores = E @ g

print("E shape:", E.shape)
print("g shape:", g.shape)
print("scores shape:", scores.shape)

k = 20

candidate_ids = torch.topk(
    -scores,
    k=k
).indices

for token_id in candidate_ids:
  print(
      token_id.item(),
      repr(tokenizer.decode([token_id.item()])),
      scores[token_id].item()
  )

E shape: torch.Size([50257, 768])
g shape: torch.Size([768])
scores shape: torch.Size([50257])
4077 ' green' -2.7974319458007812
7872 ' yellow' -2.592972993850708
4171 ' blue' -2.2200815677642822
14809 'green' -2.133229970932007
2266 ' red' -2.1200788021087646
17585 'blue' -2.1191296577453613
3469 ' Green' -2.118635416030884
445 'red' -2.074859142303467
12550 ' Yellow' -1.9714281558990479
7586 ' brown' -1.9071260690689087
39499 'Yellow' -1.8711154460906982
13719 'Green' -1.793087124824524
47606 ' GREEN' -1.7693560123443604
30966 ' greens' -1.7476320266723633
4518 ' Blue' -1.7391771078109741
36022 'yellow' -1.7385504245758057
11398 ' pink' -1.6498243808746338
14573 'Blue' -1.6347702741622925
14032 ' purple' -1.6078367233276367
23848 ' RED' -1.59745192527771


In [12]:
results = []

for token_id in candidate_ids:
  candidate = full_ids.clone()

  candidate[0, edit_position] = token_id

  candidate_loss = get_target_loss(candidate)

  results.append(
      (
          candidate_loss,
          token_id.item(),
          scores[token_id].item()
      )
  )

results.sort()

In [13]:
for candidate_loss, token_id, proxy_score in results:
    print(
        repr(tokenizer.decode([token_id])),
        "proxy =", round(proxy_score, 3),
        "actual_loss =", round(candidate_loss, 3)
    )

'Blue' proxy = -1.635 actual_loss = 7.044
'Green' proxy = -1.793 actual_loss = 7.13
' Blue' proxy = -1.739 actual_loss = 7.196
' Green' proxy = -2.119 actual_loss = 7.695
'Yellow' proxy = -1.871 actual_loss = 7.861
' Yellow' proxy = -1.971 actual_loss = 8.44
'green' proxy = -2.133 actual_loss = 8.493
'blue' proxy = -2.119 actual_loss = 9.062
'yellow' proxy = -1.739 actual_loss = 9.086
' green' proxy = -2.797 actual_loss = 9.271
'red' proxy = -2.075 actual_loss = 9.537
' RED' proxy = -1.597 actual_loss = 9.711
' blue' proxy = -2.22 actual_loss = 9.966
' yellow' proxy = -2.593 actual_loss = 10.289
' GREEN' proxy = -1.769 actual_loss = 10.303
' purple' proxy = -1.608 actual_loss = 10.503
' pink' proxy = -1.65 actual_loss = 10.56
' red' proxy = -2.12 actual_loss = 10.567
' brown' proxy = -1.907 actual_loss = 11.471
' greens' proxy = -1.748 actual_loss = 12.708


In [14]:
best_loss, best_token_id, best_proxy = results[0]

old_token = tokenizer.decode([full_ids[0, edit_position].item()])

if best_loss < baseline_loss:
  full_ids[0, edit_position] = best_token_id
  baseline_loss = best_loss

print("old token:", repr(old_token))
print("new token:", repr(tokenizer.decode([best_token_id])))
print("new loss:", baseline_loss)
print("new full prompt:", repr(tokenizer.decode(full_ids[0])))

old token: ' red'
new token: 'Blue'
new loss: 7.044469356536865
new full prompt: 'The capital of France is blue greenBlue'


In [15]:
edit_position = suffix_start + 1

print(
    "editing:",
    repr(tokenizer.decode([full_ids[0, edit_position].item()]))
)

editing: ' green'


In [16]:
edit_position = suffix_start + 1

inputs_embeds = embedding_layer(full_ids).detach()
inputs_embeds.requires_grad_(True)

outputs = model(inputs_embeds=inputs_embeds)

next_token_logits = outputs.logits[:, -1, :]

loss = F.cross_entropy(
    next_token_logits,
    target_ids[:, 0]
)

loss.backward()

g = inputs_embeds.grad[0, edit_position]

print("editing:", repr(tokenizer.decode(
    [full_ids[0, edit_position].item()]
)))
print("current loss:", loss.item())
print("g shape:", g.shape)

editing: ' green'
current loss: 7.044469356536865
g shape: torch.Size([768])


In [17]:
E = embedding_layer.weight.detach()

scores = E @ g

k = 20

candidate_ids = torch.topk(
    -scores,
    k=k
).indices

for token_id in candidate_ids:
  print(
      token_id.item(),
      repr(tokenizer.decode([token_id.item()])),
      scores[token_id].item()
  )

11427 ' Pant' -2.0158400535583496
45766 'Iv' -1.824143648147583
42251 ' roofs' -1.679648995399475
27291 ' rooft' -1.6643714904785156
20604 ' Parks' -1.6245312690734863
15857 ' pant' -1.5841050148010254
23280 ' Plaza' -1.5810558795928955
593 'own' -1.5452585220336914
48542 'wark' -1.5441036224365234
47516 'nw' -1.5167419910430908
7673 'ouch' -1.51499342918396
7642 'andal' -1.501989722251892
20593 'ogyn' -1.4865773916244507
37544 ' Blanc' -1.4410936832427979
28383 'skirts' -1.4403213262557983
27717 'seys' -1.4399738311767578
8765 ' Tower' -1.434952974319458
43076 'paces' -1.4229397773742676
43960 'CENT' -1.4209843873977661
49416 ' Loft' -1.4098143577575684


In [18]:
current_token_id = full_ids[0, edit_position].item()

candidate_pool = candidate_ids.tolist()

if current_token_id not in candidate_pool:
  candidate_pool.append(current_token_id)

results = []

for token_id in candidate_pool:
  candidate = full_ids.clone()

  candidate[0, edit_position] = token_id

  candidate_loss = get_target_loss(candidate)

  results.append(
      (
          candidate_loss,
          token_id,
          scores[token_id].item()
      )
  )

results.sort()

In [19]:
for candidate_loss, token_id, proxy_score in results:
    print(
        repr(tokenizer.decode([token_id])),
        "proxy =", round(proxy_score, 3),
        "actual_loss =", round(candidate_loss, 3)
    )

' Parks' proxy = -1.625 actual_loss = 5.923
' Plaza' proxy = -1.581 actual_loss = 6.299
' rooft' proxy = -1.664 actual_loss = 6.383
' Tower' proxy = -1.435 actual_loss = 6.651
' Pant' proxy = -2.016 actual_loss = 6.876
' green' proxy = 0.369 actual_loss = 7.044
'ouch' proxy = -1.515 actual_loss = 7.133
' pant' proxy = -1.584 actual_loss = 7.167
'skirts' proxy = -1.44 actual_loss = 7.307
'paces' proxy = -1.423 actual_loss = 7.31
' Blanc' proxy = -1.441 actual_loss = 7.534
' roofs' proxy = -1.68 actual_loss = 7.575
'CENT' proxy = -1.421 actual_loss = 7.682
' Loft' proxy = -1.41 actual_loss = 7.926
'Iv' proxy = -1.824 actual_loss = 8.018
'ogyn' proxy = -1.487 actual_loss = 8.254
'own' proxy = -1.545 actual_loss = 9.295
'seys' proxy = -1.44 actual_loss = 9.327
'wark' proxy = -1.544 actual_loss = 10.037
'andal' proxy = -1.502 actual_loss = 10.209
'nw' proxy = -1.517 actual_loss = 11.582


In [20]:
best_loss, best_token_id, best_proxy = results[0]

full_ids[0, edit_position] = best_token_id
baseline_loss = best_loss

print("new suffix/full prompt:", repr(tokenizer.decode(full_ids[0])))
print("new loss:", baseline_loss)

new suffix/full prompt: 'The capital of France is blue ParksBlue'
new loss: 5.922564506530762


In [21]:
editable_positions = list(range(suffix_start, full_ids.shape[1]))

inputs_embeds = embedding_layer(full_ids).detach()
inputs_embeds.requires_grad_(True)

outputs = model(inputs_embeds=inputs_embeds)
next_token_logits = outputs.logits[:, -1, :]

loss = F.cross_entropy(
    next_token_logits,
    target_ids[:, 0]
)

loss.backward()

grads = inputs_embeds.grad[0, editable_positions]

print("editable postiions:", editable_positions)
print("grads shape:", grads.shape)

editable postiions: [5, 6, 7]
grads shape: torch.Size([3, 768])


In [22]:
E = embedding_layer.weight.detach()

k = 20

topk_per_position = {}

for local_idx, position in enumerate(editable_positions):
  g = grads[local_idx]

  scores = E @ g

  candidate_ids = torch.topk(
      -scores,
      k=k
  ).indices

  topk_per_position[position] = candidate_ids

In [23]:
import random

B = 30

candidates = []

for _ in range(B):
  position = random.choice(editable_positions)

  token_id = random.choice(
      topk_per_position[position].tolist()
  )

  candidate = full_ids.clone()
  candidate[0, position] = token_id

  candidates.append((candidate, position, token_id))

In [24]:
evaluated = []

for candidate, position, token_id in candidates:
  candidate_loss = get_target_loss(candidate)

  evaluated.append(
      (
          candidate_loss,
          candidate,
          position,
          token_id
      )
  )

evaluated.sort(key=lambda x: x[0])

In [25]:
for candidate_loss, candidate, position, token_id in evaluated[:10]:
    print(
        "loss =", round(candidate_loss, 3),
        "position =", position,
        "token =", repr(tokenizer.decode([token_id])),
        "prompt =", repr(tokenizer.decode(candidate[0]))
    )

loss = 4.81 position = 7 token = ' red' prompt = 'The capital of France is blue Parks red'
loss = 5.836 position = 7 token = ' N' prompt = 'The capital of France is blue Parks N'
loss = 5.837 position = 5 token = ' LGBT' prompt = 'The capital of France is LGBT ParksBlue'
loss = 6.144 position = 6 token = ' Orche' prompt = 'The capital of France is blue OrcheBlue'
loss = 6.144 position = 6 token = ' Orche' prompt = 'The capital of France is blue OrcheBlue'
loss = 6.287 position = 7 token = ' make' prompt = 'The capital of France is blue Parks make'
loss = 6.3 position = 7 token = ' left' prompt = 'The capital of France is blue Parks left'
loss = 6.334 position = 6 token = ' architect' prompt = 'The capital of France is blue architectBlue'
loss = 6.475 position = 7 token = ' beauty' prompt = 'The capital of France is blue Parks beauty'
loss = 6.512 position = 6 token = ' Claudia' prompt = 'The capital of France is blue ClaudiaBlue'


In [26]:
best_loss, best_candidate, best_position, best_token_id = evaluated[0]

if best_loss < baseline_loss:
    full_ids = best_candidate.clone()
    baseline_loss = best_loss

print("new loss:", baseline_loss)
print("new prompt:", repr(tokenizer.decode(full_ids[0])))
print(
    "accepted edit:",
    best_position,
    repr(tokenizer.decode([best_token_id]))
)

new loss: 4.810265064239502
new prompt: 'The capital of France is blue Parks red'
accepted edit: 7 ' red'


Clean GCG Implementation:

In [35]:
def get_candidate_loss(candidate_ids, target_id):
    if candidate_ids.dim() == 1:
        candidate_ids = candidate_ids.unsqueeze(0)

    with torch.no_grad():
        outputs = model(input_ids=candidate_ids)

        next_token_logits = outputs.logits[:, -1, :]

        loss = F.cross_entropy(
            next_token_logits,
            target_id
        )

    return loss.item()

def get_gradient_at_positions(candidate_ids, target_id, editable_positions):
    embeddings_library = model.get_input_embeddings()

    embeddings_for_sequence = embeddings_library(candidate_ids)
    detached_embeddings = embeddings_for_sequence.detach()
    detached_embeddings.requires_grad_(True)
    outputs = model(inputs_embeds=detached_embeddings)

    logits = outputs.logits

    next_token_logits = logits[:, -1, :]

    loss =F.cross_entropy(
        next_token_logits,
        target_id
    )

    loss.backward()

    g = detached_embeddings.grad[0, editable_positions]

    return g

def get_top_k(gradients, k):
    embeddings_library = model.get_input_embeddings().weight.detach()

    proxy_scores = gradients @ embeddings_library.T

    top_k = torch.topk(
        -proxy_scores,
        k=k,
        dim=1
    ).indices

    return top_k

def batch_of_candidates(candidate_ids, editable_positions, top_k, B):
    batch_of_candidates = []

    for b in range(B):
      local_idx = random.choice(range(len(editable_positions)))

      chosen_edit_pos = editable_positions[local_idx]
      chosen_top_k = random.choice(top_k[local_idx].tolist())

      new_candidate = candidate_ids.clone()
      new_candidate[0, chosen_edit_pos] = chosen_top_k

      batch_of_candidates.append(new_candidate)
    return torch.cat(batch_of_candidates, dim=0)

def run_gcg(candidate_ids, target_id, editable_positions,
            num_iterations=10, k=20, B=30):
  if candidate_ids.dim() == 1:
    candidate_ids = candidate_ids.unsqueeze(0)

    for iteration in range(num_iterations):

        curr_loss = get_candidate_loss(candidate_ids, target_id)

        gradients = get_gradient_at_positions(
            candidate_ids,
            target_id,
            editable_positions
        )

        top_k = get_top_k(gradients, k)

        candidate_batch = batch_of_candidates(
            candidate_ids,
            editable_positions,
            top_k,
            B
        )

        min_loss = None
        min_candidate = None

        for candidate in candidate_batch:

            # Restore batch dimension: [seq] -> [1, seq]
            candidate = candidate.unsqueeze(0)

            loss = get_candidate_loss(candidate, target_id)

            if min_loss is None or loss < min_loss:
                min_loss = loss
                min_candidate = candidate

        if min_loss < curr_loss:
            candidate_ids = min_candidate

        print(
            iteration,
            "loss:", min(curr_loss, min_loss),
            "prompt:", repr(tokenizer.decode(candidate_ids[0]))
        )

    return candidate_ids


Comparison to random selection of candidates without gradient guide:

In [36]:
def run_gcg_random(candidate_ids, target_id, editable_positions,
            num_iterations=10, k=20, B=30):

  if candidate_ids.dim() == 1:
    candidate_ids = candidate_ids.unsqueeze(0)

    for iteration in range(num_iterations):

        curr_loss = get_candidate_loss(candidate_ids, target_id)

        vocab_size = model.get_input_embeddings().weight.shape[0]

        top_k = torch.randint(
            0,
            vocab_size,
            (len(editable_positions), k)
        )

        candidate_batch = batch_of_candidates(
            candidate_ids,
            editable_positions,
            top_k,
            B
        )

        min_loss = None
        min_candidate = None

        for candidate in candidate_batch:

            # Restore batch dimension: [seq] -> [1, seq]
            candidate = candidate.unsqueeze(0)

            loss = get_candidate_loss(candidate, target_id)

            if min_loss is None or loss < min_loss:
                min_loss = loss
                min_candidate = candidate

        if min_loss < curr_loss:
            candidate_ids = min_candidate

        print(
            iteration,
            "loss:", min(curr_loss, min_loss),
            "prompt:", repr(tokenizer.decode(candidate_ids[0]))
        )

    return candidate_ids

In [38]:
seeds = [0, 1, 2]
results = []

original_start_ids = start_ids.clone()

for seed in seeds:
    # --- GCG ---
    random.seed(seed)
    torch.manual_seed(seed)

    gcg_result = run_gcg(
        original_start_ids.clone(),
        target_id,
        editable_positions,
        num_iterations=10,
        k=20,
        B=30
    )

    # Reset seed so random baseline gets a reproducible run too
    random.seed(seed)
    torch.manual_seed(seed)

    random_result = run_gcg_random(
        original_start_ids.clone(),
        target_id,
        editable_positions,
        num_iterations=10,
        k=20,
        B=30
    )

    initial_loss = get_candidate_loss(original_start_ids, target_id)
    gcg_loss = get_candidate_loss(gcg_result, target_id)
    random_loss = get_candidate_loss(random_result, target_id)

    results.append(
        (seed, initial_loss, gcg_loss, random_loss)
    )

0 loss: 5.1740217208862305 prompt: ' red make color Clcolor wood Paris design " NSand made clean change J beauty fill leftented look'
1 loss: 4.430787086486816 prompt: ' red make color Clcolor sprite Paris design " NSand made clean change J beauty fill leftented look'
2 loss: 3.847588062286377 prompt: ' red make color Clcolor fill Paris design " NSand made clean change J beauty fill leftented look'
3 loss: 3.0043389797210693 prompt: ' red make color Clcolor fill Paris description " NSand made clean change J beauty fill leftented look'
4 loss: 2.9789812564849854 prompt: ' red make color Clcolor fill ParisDesktop " NSand made clean change J beauty fill leftented look'
5 loss: 2.9789812564849854 prompt: ' red make color Clcolor fill ParisDesktop " NSand made clean change J beauty fill leftented look'
6 loss: 2.9789812564849854 prompt: ' red make color Clcolor fill ParisDesktop " NSand made clean change J beauty fill leftented look'
7 loss: 2.9789812564849854 prompt: ' red make color Clcol

In [39]:
print("\n=== 3-SEED COMPARISON ===")
print(f"{'Seed':<6}{'Initial':<12}{'GCG':<12}{'Random':<12}")

for seed, initial, gcg, rand in results:
    print(
        f"{seed:<6}"
        f"{initial:<12.3f}"
        f"{gcg:<12.3f}"
        f"{rand:<12.3f}"
    )


=== 3-SEED COMPARISON ===
Seed  Initial     GCG         Random      
0     10.130      2.438       7.464       
1     10.130      3.597       6.783       
2     10.130      3.466       5.553       


In [40]:
gcg_losses = [row[2] for row in results]
random_losses = [row[3] for row in results]

print("\nAverage final loss")
print("GCG:", round(sum(gcg_losses) / len(gcg_losses), 3))
print("Random:", round(sum(random_losses) / len(random_losses), 3))


Average final loss
GCG: 3.167
Random: 6.6


**Question**: Does gradient-guided discrete token search find lower-target-loss suffixes than random token search under the same candidate-evaluation budget?

**Method**: I used GPT-2, and sought to elicit a benign " Paris" target given the prompt "The Capital of France is", an editable suffix, and Hugging Face's embedding gradients. I then turned those embeddings into a vocabulary proxy ranking → sampled discrete candidates → actual CE evaluation → greedy update the best loss candidate.

**Paper connection**: GCG is an algorithm which utilizes a white-box model and finds candidates using gradients to reduce the search space of candidates to select the best replacement token from. We iteratively sub in the candidates with the best loss to systematically create suffixes that increase the possibility of the target response being generated.

**Experiment**: held constant: same start, iterations, k, and B; 3 random seeds; variables: GCG versus random candidate proposals.

**Result**: Compared it to a random instead of gradient driven selector to reduce search space. The average final loss 3.167 GCG vs 6.600 random. This is a small toy experiment however and thus too small to draw statistical significance.

**Limitations**: I implemented a toy version of the paper's algorithm so was limited to a single model, single benign target, only 3 seeds, embedding-space approximation instead of the paper's exact one-hot formulation, and equal forward-evaluation budget rather than equal total compute because GCG also pays for backward passes.

**What I learned**: I learned how to do discrete optimization, autograd through LMs, proxy objective vs behavioral outcome, causal transformer gradients, and how to decompose a paper algorithm into implementation primitives.

**Next experiment**: Implement Algorithm 2 from GCG paper